# Nestle Valuation Case Study

This case study combines DCF valuation, relative valuation, and sensitivity analysis for a Nestle-like company using synthetic learning inputs.

Abbreviations used in this notebook:

- **DCF**: Discounted Cash Flow.
- **FCF**: Free Cash Flow.
- **WACC**: Weighted Average Cost of Capital.
- **EV**: Enterprise Value.
- **TV**: Terminal Value.
- **P/E**: Price to Earnings.
- **EV/EBITDA**: Enterprise Value divided by EBITDA.
- **CHF**: Swiss franc.

## 1. Intuition

A complete valuation case should not rely on one method. DCF estimates intrinsic value from cash flows, while relative valuation checks what the market pays for comparable companies. Sensitivity analysis shows how fragile the conclusion is.

## 2. Mathematics

**DCF enterprise value:**

$$
EV = \sum_{t=1}^{N}\frac{FCF_t}{(1+WACC)^t} + \frac{TV}{(1+WACC)^N}
$$

Where:

- $FCF_t$ = free cash flow in period $t$
- $WACC$ = weighted average cost of capital
- $TV$ = terminal value
- $EV$ = enterprise value, the value of the operating business
- $t$ = time period index

**Terminal value:**

$$
TV = \frac{FCF_N(1+g)}{WACC-g}
$$

Where:

- $FCF_N$ = free cash flow in the final explicit forecast year
- $WACC$ = weighted average cost of capital
- $TV$ = terminal value
- $g$ = long-term growth rate

**Equity value per share:**

$$
Value/Share = \frac{EV - NetDebt}{Shares}
$$

Where:

- $EV$ = enterprise value, the value of the operating business
- $\text{Equity Value}$ = value attributable to shareholders
- $\text{Net Debt}$ = debt minus cash and cash equivalents
- $\text{Value per Share}$ = intrinsic equity value per share
- $\text{Shares Outstanding}$ = number of shares issued and outstanding

**Relative value from P/E:**

$$
Value/Share = EPS \times Peer\ P/E
$$

Where:

- $\text{Equity Value}$ = value attributable to shareholders
- $\text{Value per Share}$ = intrinsic equity value per share
- $EPS$ = earnings per share
- $P/E$ = price-to-earnings multiple

## 3. Implementation

We run a base DCF, compare it with peer-implied valuation, and estimate upside or downside versus an illustrative market price.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

assumptions = {
    "fcf0": 12_300,
    "growth": 0.035,
    "wacc": 0.060,
    "terminal_growth": 0.020,
    "years": 5,
    "net_debt": 35_000,
    "shares": 2_650,
    "current_price": 96.0,
}

def dcf_value(fcf0, growth, wacc, terminal_growth, years, net_debt, shares):
    fcfs = np.array([fcf0 * (1 + growth) ** year for year in range(1, years + 1)])
    pv_fcfs = np.array([fcfs[year-1] / (1 + wacc) ** year for year in range(1, years + 1)])
    tv = fcfs[-1] * (1 + terminal_growth) / (wacc - terminal_growth)
    pv_tv = tv / (1 + wacc) ** years
    ev = pv_fcfs.sum() + pv_tv
    equity = ev - net_debt
    return pd.Series({"pv_fcfs": pv_fcfs.sum(), "pv_terminal": pv_tv, "enterprise_value": ev, "equity_value": equity, "value_per_share": equity / shares, "terminal_value_share": pv_tv / ev})

valuation = dcf_value(**{k: assumptions[k] for k in ["fcf0", "growth", "wacc", "terminal_growth", "years", "net_debt", "shares"]})
valuation.to_frame("base_case").round(2)

In [ ]:
peers = pd.DataFrame({
    "company": ["Peer A", "Peer B", "Peer C", "Peer D", "Peer E"],
    "pe": [20.5, 22.0, 18.8, 21.4, 19.7],
    "ev_ebitda": [13.2, 14.0, 12.4, 13.7, 12.9],
})
net_income = 12_100
ebitda = 22_400
eps = net_income / assumptions["shares"]
peer_pe_value = eps * peers["pe"].median()
peer_ev_value = (ebitda * peers["ev_ebitda"].median() - assumptions["net_debt"]) / assumptions["shares"]
relative_values = pd.Series({"P/E implied price": peer_pe_value, "EV/EBITDA implied price": peer_ev_value})
relative_values.to_frame("CHF_per_share").round(2)

## 4. Visualization

The valuation output should show the base-case gap, the method range, and sensitivity to WACC and terminal growth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.Series({"Current price": assumptions["current_price"], "DCF": valuation["value_per_share"], **relative_values.to_dict()}).plot(kind="bar", ax=axes[0], color="#2f6f8f")
axes[0].set_title("Valuation Methods")
axes[0].set_ylabel("CHF per share")
axes[0].tick_params(axis="x", rotation=25)

valuation[["pv_fcfs", "pv_terminal"]].plot(kind="bar", ax=axes[1], color="#9a6b2f")
axes[1].set_title("DCF Enterprise Value Components")
axes[1].set_ylabel("CHF millions")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
wacc_values = np.linspace(0.05, 0.075, 6)
growth_values = np.linspace(0.01, 0.03, 6)
sensitivity = pd.DataFrame(index=wacc_values, columns=growth_values, dtype=float)
for w in wacc_values:
    for g in growth_values:
        sensitivity.loc[w, g] = dcf_value(assumptions["fcf0"], assumptions["growth"], w, g, assumptions["years"], assumptions["net_debt"], assumptions["shares"])["value_per_share"]

fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(sensitivity.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(growth_values))); ax.set_xticklabels([f"{g:.1%}" for g in growth_values])
ax.set_yticks(range(len(wacc_values))); ax.set_yticklabels([f"{w:.1%}" for w in wacc_values])
ax.set_xlabel("Terminal growth"); ax.set_ylabel("WACC"); ax.set_title("DCF Sensitivity")
for r in range(sensitivity.shape[0]):
    for c in range(sensitivity.shape[1]):
        ax.text(c, r, f"{sensitivity.values[r,c]:.0f}", ha="center", va="center", color="white", fontsize=8)
fig.colorbar(image, ax=ax, label="CHF per share")
plt.tight_layout(); plt.show()

## 5. Application

The base DCF and relative valuation should be synthesized into an investment range. If DCF value is above market price but peer valuation is closer to the market price, the conclusion should emphasize assumption risk.

In [ ]:
valuation_range = pd.Series({
    "low": min(relative_values.min(), sensitivity.min().min()),
    "base_dcf": valuation["value_per_share"],
    "high": max(relative_values.max(), sensitivity.max().max()),
})
upside = valuation["value_per_share"] / assumptions["current_price"] - 1
print(f"Base DCF value per share: CHF {valuation['value_per_share']:.2f}")
print(f"Illustrative upside/downside: {upside:.1%}")
print(f"Terminal value share of EV: {valuation['terminal_value_share']:.1%}")
valuation_range.to_frame("CHF_per_share")

## 6. Reflection

- DCF and multiples answer different valuation questions.
- Terminal value dominates mature-company valuation.
- A valuation range is more honest than a single price target.
- The final conclusion should depend on both value and business quality.

Questions to answer after running the notebook:

1. Does DCF agree with peer valuation?
2. Which assumption drives the largest sensitivity?
3. Is the terminal value share acceptable?
4. Would this case support a buy, hold, or avoid conclusion?